# TP3 - Procesamiento del Lenguaje Natural (NLP)
**Inteligencia Artificial y Neurociencias**

## 2. Modelos clásicos de NLP

En esta parte se construye un **baseline** que combina una representación textual (TF-IDF) con un clasificador lineal, y luego se experimenta con variantes de **preprocesamiento**, **entrada del modelo**, **features**, **clasificadores** e **hiperparámetros** para intentar mejorarlo.

**Protocolo de evaluación** (definido en la parte 1):
- Se usan exactamente los conjuntos de `splits_tp3.csv` (split agrupado por propuesta, ≈70/15/15).
- Todas las decisiones se toman con **validación** (y con validación cruzada agrupada sobre train+val en la elección final). El conjunto de **test se usa una única vez**, al final (sección 2.8), para el baseline y el modelo elegido.
- Métrica principal: **Macro-F1** respecto de `majority_label` (clases *favor*, *against*, *neutral*). Se reportan además accuracy, F1 por clase, tiempos de entrenamiento e inferencia.

**Estrategia de búsqueda:** las decisiones se toman de forma secuencial (*greedy*): primero el manejo del desbalance, luego el preprocesamiento, la entrada, las features y por último el clasificador y sus hiperparámetros. Cada paso parte de la mejor configuración del paso anterior. No es una búsqueda exhaustiva (algunas combinaciones no se prueban), pero mantiene el número de experimentos acotado y cada efecto es interpretable por separado.

> **Para correr en Colab:** subir `splits_tp3.csv` (generado por la parte 1) al entorno, o ejecutar la celda de carga y subirlo cuando lo pida. No requiere GPU.

### 2.0. Configuración

Se instalan e importan las dependencias. spaCy viene preinstalado en Colab; solo hace falta descargar el modelo de español para la lematización.

In [ ]:
!pip -q install datasets
!python -m spacy download es_core_news_sm -q

In [ ]:
import os
import re
import time
import pickle
import platform
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

import nltk
import spacy
import sklearn
from nltk.corpus import stopwords
from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

nltk.download("stopwords", quiet=True)

SEED = 42
CLASES = ["favor", "against", "neutral"]   # mismo orden que en la parte 1
label2id = {c: i for i, c in enumerate(CLASES)}
id2label = {i: c for c, i in label2id.items()}
sns.set_theme(style="whitegrid")
np.random.seed(SEED)

# Hardware / versiones (para reportar costos)
print("Python:", platform.python_version(), "| scikit-learn:", sklearn.__version__, "| spaCy:", spacy.__version__)
print("CPU:", platform.processor() or platform.machine(), "| núcleos:", os.cpu_count())

### 2.1. Carga de datos y particiones

Se reconstruye `df_modelado` igual que en la parte 1 (se descartan los 150 comentarios sin mayoría) y se le asigna a cada comentario su conjunto según `splits_tp3.csv`. Se verifica que todos los comentarios tengan conjunto asignado y que ninguna propuesta aparezca en más de un conjunto.

In [ ]:
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")
df = dataset["train"].to_pandas()

cols_anotadores = ["annotation_1", "annotation_2", "annotation_3"]
df_modelado = df[df["majority_label"].isin(CLASES)].copy().reset_index(drop=True)
df_modelado["label"] = df_modelado["majority_label"]
df_modelado["id"] = df_modelado["id"].astype(str)

# Nivel de acuerdo entre anotadores (se usa en el análisis de errores)
df_modelado["max_acuerdo"] = df_modelado[cols_anotadores].apply(lambda f: f.value_counts().max(), axis=1)
df_modelado["nivel_acuerdo"] = df_modelado["max_acuerdo"].map({3: "Total (3/3)", 2: "Parcial (2/3)"})

RUTA_SPLITS = "splits_tp3.csv"
if not os.path.exists(RUTA_SPLITS):
    try:
        from google.colab import files
        print("No se encontró splits_tp3.csv: subilo (lo genera la parte 1).")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Falta splits_tp3.csv (lo genera la parte 1).")

splits = pd.read_csv(RUTA_SPLITS, dtype={"id": str})
df_modelado = df_modelado.merge(splits, on="id", how="left", validate="one_to_one")
assert df_modelado["split"].notna().all(), "Hay comentarios sin split asignado"

train_df = df_modelado[df_modelado["split"] == "train"].reset_index(drop=True)
val_df = df_modelado[df_modelado["split"] == "val"].reset_index(drop=True)
test_df = df_modelado[df_modelado["split"] == "test"].reset_index(drop=True)

# Controles: todos los comentarios tienen split y no hay propuestas compartidas
assert len(train_df) + len(val_df) + len(test_df) == len(df_modelado) == 2850
print(f"Train: {len(train_df)} | Validación: {len(val_df)} | Test: {len(test_df)}")
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert not set(a["target"]) & set(b["target"])

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

display(pd.crosstab(df_modelado["split"], df_modelado["label"]).loc[["train", "val", "test"], CLASES])

### 2.2. Preprocesamiento y herramientas de evaluación

Siguiendo el plan de la sección 1.7, se definen las variantes de preprocesamiento a comparar:

| Variante | Qué hace |
|---|---|
| `original` | Texto sin modificar (se respetan mayúsculas) |
| `minusculas` | Tokenización + minúsculas |
| `sin_acentos` | Minúsculas + eliminación de tildes |
| `sin_stopwords` | Minúsculas + eliminación de todas las stopwords de NLTK |
| `sin_stopwords_conserva_neg` | Igual, pero conservando las negaciones (*no*, *ni*, *sin*, *nunca*...) |
| `lemas` | Lematización con spaCy (`es_core_news_sm`) + minúsculas |
| `lemas_sin_stop_neg` | Lemas + eliminación de stopwords conservando negaciones |

Los textos preprocesados se calculan una sola vez y se guardan en caché. La lematización es el único paso con costo de cómputo apreciable, por lo que se mide su tiempo por separado.

In [ ]:
stop_es = set(stopwords.words("spanish"))
NEGACIONES = {"no", "ni", "sin", "nunca", "tampoco", "nada", "nadie", "jamás",
              "ningún", "ninguna", "ninguno"}
stop_sin_neg = stop_es - NEGACIONES
print("Negaciones que estaban en la lista de stopwords:", sorted(NEGACIONES & stop_es))

RE_TOKEN = re.compile(r"\w+")
TOKEN_PATTERN = r"(?u)\b\w+\b"   # conserva tokens de 1 letra (p. ej. 'y', 'o')

def sin_acentos(t):
    return "".join(ch for ch in unicodedata.normalize("NFD", t) if unicodedata.category(ch) != "Mn")

nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
_cache_lemas = {}

def lematizar(textos):
    pendientes = [t for t in pd.unique(pd.Series(textos)) if t not in _cache_lemas]
    for t, doc in zip(pendientes, nlp.pipe(pendientes, batch_size=64)):
        _cache_lemas[t] = [tok.lemma_.lower() for tok in doc if not (tok.is_space or tok.is_punct)]
    return [_cache_lemas[t] for t in textos]

VARIANTES = {
    "original":                   dict(minus=False, acentos=True,  lemas=False, stop=None),
    "minusculas":                 dict(minus=True,  acentos=True,  lemas=False, stop=None),
    "sin_acentos":                dict(minus=True,  acentos=False, lemas=False, stop=None),
    "sin_stopwords":              dict(minus=True,  acentos=True,  lemas=False, stop=stop_es),
    "sin_stopwords_conserva_neg": dict(minus=True,  acentos=True,  lemas=False, stop=stop_sin_neg),
    "lemas":                      dict(minus=True,  acentos=True,  lemas=True,  stop=None),
    "lemas_sin_stop_neg":         dict(minus=True,  acentos=True,  lemas=True,  stop=stop_sin_neg),
}

def preprocesar(textos, variante):
    cfg = VARIANTES[variante]
    textos = list(textos)
    if variante == "original":
        return textos
    if cfg["lemas"]:
        toks = lematizar(textos)
    else:
        toks = [RE_TOKEN.findall(t.lower() if cfg["minus"] else t) for t in textos]
    if cfg["stop"] is not None:
        toks = [[w for w in ts if w not in cfg["stop"]] for ts in toks]
    if not cfg["acentos"]:
        toks = [[sin_acentos(w) for w in ts] for ts in toks]
    return [" ".join(ts) for ts in toks]

# Ejemplo
ejemplo = train_df["comment"].iloc[3]
print("ORIGINAL:", ejemplo[:200], "\n")
for v in ["sin_stopwords", "sin_stopwords_conserva_neg", "lemas_sin_stop_neg"]:
    print(f"{v}:", preprocesar([ejemplo], v)[0][:200], "\n")

In [ ]:
# Costo de la lematización: se lematizan una vez todos los comentarios y títulos de propuesta
todos = pd.concat([df_modelado["comment"], df_modelado["target"]]).unique().tolist()
_cache_lemas.clear()
t0 = time.perf_counter()
lematizar(todos)
T_LEMATIZACION = time.perf_counter() - t0
print(f"Lematización de {len(todos)} textos únicos: {T_LEMATIZACION:.1f} s "
      f"({T_LEMATIZACION / len(todos) * 1000:.1f} ms por texto)")

**Entrada del modelo.** Como vimos en la parte 1, la postura depende de la propuesta (el mismo texto puede ser *favor* en una propuesta y *against* en otra). Se comparan tres formas de armar la entrada:
- `comentario`: solo el comentario.
- `concatenado`: título de la propuesta + comentario en un mismo texto (comparten vocabulario).
- `separado`: un vectorizador para el título y otro para el comentario (features independientes).

No se usa la `description` completa porque es muy larga (mediana de 105 palabras) y es casi siempre la misma dentro de una propuesta: en bolsa de palabras aportaría sobre todo vocabulario temático.

A continuación se definen las funciones que arman la entrada, construyen el pipeline (vectorizador + clasificador), y entrenan, evalúan y registran cada experimento.

In [ ]:
DFS = {"train": train_df, "val": val_df, "test": test_df}
_cache_prep = {}

def textos(split, campo, variante):
    clave = (split, campo, variante)
    if clave not in _cache_prep:
        _cache_prep[clave] = preprocesar(DFS[split][campo], variante)
    return _cache_prep[clave]

def armar_X(split, variante, entrada):
    com = textos(split, "comment", variante)
    if entrada == "comentario":
        return com
    prop = textos(split, "target", variante)
    if entrada == "concatenado":
        return [p + " " + c for p, c in zip(prop, com)]
    if entrada == "separado":
        return pd.DataFrame({"prop": prop, "com": com})
    raise ValueError(entrada)

def unir(a, b):
    return pd.concat([a, b], ignore_index=True) if isinstance(a, pd.DataFrame) else list(a) + list(b)

def tomar(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else [X[i] for i in idx]

def vectorizador(cfg_feat):
    cfg = dict(cfg_feat)
    tipo = cfg.pop("tipo")
    if tipo == "bow":
        return CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, **cfg)
    if tipo == "tfidf":
        return TfidfVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, **cfg)
    if tipo == "char":
        return TfidfVectorizer(analyzer="char_wb", lowercase=False, **cfg)
    if tipo == "word+char":
        return FeatureUnion([
            ("word", TfidfVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False,
                                     ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
            ("char", TfidfVectorizer(analyzer="char_wb", lowercase=False,
                                     ngram_range=(2, 5), sublinear_tf=True, min_df=2)),
        ])
    raise ValueError(tipo)

def armar_pipeline(config):
    if config["entrada"] == "separado":
        feats = ColumnTransformer([("prop", vectorizador(config["cfg_feat"]), "prop"),
                                   ("com", vectorizador(config["cfg_feat"]), "com")])
    else:
        feats = vectorizador(config["cfg_feat"])
    return Pipeline([("feats", feats), ("clf", clone(config["modelo"]))])

def metricas(y_true, y_pred):
    f1s = f1_score(y_true, y_pred, labels=CLASES, average=None, zero_division=0)
    return {"macro_f1": f1_score(y_true, y_pred, labels=CLASES, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1s)}}

resultados = []   # registro de todos los experimentos sobre validación

def experimento(grupo, nombre, variante, entrada, cfg_feat, modelo):
    config = dict(variante=variante, entrada=entrada, cfg_feat=cfg_feat, modelo=clone(modelo))
    X_tr, X_va = armar_X("train", variante, entrada), armar_X("val", variante, entrada)
    pipe = armar_pipeline(config)
    t0 = time.perf_counter(); pipe.fit(X_tr, y_train); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); y_pred = pipe.predict(X_va); t_inf = time.perf_counter() - t0
    try:
        n_feats = len(pipe.named_steps["feats"].get_feature_names_out())
    except Exception:
        n_feats = np.nan
    fila = {"grupo": grupo, "experimento": nombre, **metricas(y_val, y_pred),
            "t_train_s": t_train, "t_inf_val_s": t_inf, "n_features": n_feats, "config": config}
    resultados.append(fila)
    return fila

def tabla(grupo):
    t = pd.DataFrame([r for r in resultados if r["grupo"] == grupo]).drop(columns=["grupo", "config"])
    return (t.sort_values("macro_f1", ascending=False).set_index("experimento")
             .style.format({c: "{:.3f}" for c in ["macro_f1", "accuracy"] + [f"f1_{x}" for x in CLASES]}
                           | {"t_train_s": "{:.2f}", "t_inf_val_s": "{:.3f}", "n_features": "{:,.0f}"})
             .background_gradient(subset=["macro_f1"], cmap="Greens"))

def mejor(grupo):
    return max((r for r in resultados if r["grupo"] == grupo), key=lambda r: r["macro_f1"])

### 2.3. Baseline

**Baseline:** TF-IDF de unigramas sobre el comentario en minúsculas (equivalente al `TfidfVectorizer` por defecto) + Regresión Logística sin ajustes. Como referencia mínima se incluyen dos clasificadores triviales: uno que siempre predice la clase mayoritaria (*favor*) y uno que predice al azar respetando las proporciones de clase.

In [ ]:
CFG_UNIGRAMAS = {"tipo": "tfidf", "ngram_range": (1, 1)}

experimento("referencia", "Dummy: clase mayoritaria", "minusculas", "comentario", CFG_UNIGRAMAS,
            DummyClassifier(strategy="most_frequent"))
experimento("referencia", "Dummy: aleatorio estratificado", "minusculas", "comentario", CFG_UNIGRAMAS,
            DummyClassifier(strategy="stratified", random_state=SEED))
BASELINE = experimento("referencia", "Baseline: TF-IDF unigramas + LogReg", "minusculas", "comentario",
                       CFG_UNIGRAMAS, LogisticRegression(max_iter=2000, random_state=SEED))
tabla("referencia")

**Observaciones** *(completar con los resultados)*

- ¿Cuánto supera el baseline a los clasificadores triviales? El Dummy mayoritario obtiene accuracy ≈ 0,6 pero Macro-F1 ≈ 0,26, lo que muestra por qué la accuracy no es una buena métrica para este problema.
- Mirar el F1 por clase del baseline: sin ajustar por desbalance, es esperable que la regresión logística prediga casi siempre *favor* y que el F1 de *neutral* (y en menor medida *against*) sea muy bajo.

### 2.4. Desbalance de clases

*favor* representa ≈62% de los datos. Se compara la regresión logística sin pesos contra `class_weight="balanced"`, que pondera cada clase en forma inversamente proporcional a su frecuencia, de modo que equivocarse en *neutral* "cuesta" más que equivocarse en *favor*.

In [ ]:
experimento("desbalance", "Sin pesos (baseline)", "minusculas", "comentario", CFG_UNIGRAMAS,
            LogisticRegression(max_iter=2000, random_state=SEED))
experimento("desbalance", "class_weight='balanced'", "minusculas", "comentario", CFG_UNIGRAMAS,
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
tabla("desbalance")

In [ ]:
LR_BAL = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)

**Observaciones** *(completar con los resultados)*

- Comparar el F1 de *neutral* y *against* con y sin pesos. Se espera una mejora fuerte del Macro-F1 a costa de algo de accuracy (y de F1 de *favor*), porque el modelo deja de "apostar" a la clase mayoritaria.
- A partir de acá todos los experimentos con regresión logística usan `class_weight="balanced"` (`LR_BAL`).

### 2.5. Variantes de preprocesamiento

Se comparan las variantes definidas en 2.2, con TF-IDF de unigramas y de uni+bigramas (las negaciones y expresiones como *"no estoy de acuerdo"* dependen del orden de las palabras, así que el efecto de eliminar stopwords puede cambiar al usar bigramas).

In [ ]:
CFG_BIGRAMAS = {"tipo": "tfidf", "ngram_range": (1, 2)}

for variante in VARIANTES:
    for nombre_ng, cfg in [("1-gramas", CFG_UNIGRAMAS), ("1-2-gramas", CFG_BIGRAMAS)]:
        experimento("preprocesamiento", f"{variante} | {nombre_ng}", variante, "comentario", cfg, LR_BAL)

tabla("preprocesamiento")

In [ ]:
# Vista resumida: Macro-F1 por variante y n-gramas
pp = pd.DataFrame([r for r in resultados if r["grupo"] == "preprocesamiento"])
pp[["variante", "ngramas"]] = pp["experimento"].str.split(" | ", regex=False, expand=True)
piv = pp.pivot(index="variante", columns="ngramas", values="macro_f1").loc[list(VARIANTES)]

ax = piv.plot.barh(figsize=(8, 5))
ax.axvline(BASELINE["macro_f1"], ls="--", c="gray", label="Baseline")
ax.set_xlabel("Macro-F1 (validación)"); ax.set_ylabel("")
ax.set_title("Efecto del preprocesamiento (TF-IDF + LogReg balanceada)")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

MEJOR_PREP = mejor("preprocesamiento")
VARIANTE = MEJOR_PREP["config"]["variante"]
print("Mejor preprocesamiento:", MEJOR_PREP["experimento"], f"→ Macro-F1 = {MEJOR_PREP['macro_f1']:.3f}")

**Observaciones** *(completar con los resultados)*

- **Stopwords:** comparar `sin_stopwords` vs. `sin_stopwords_conserva_neg`. Si eliminar las negaciones empeora el resultado (sobre todo el F1 de *against*), confirma la hipótesis de la parte 1 de que las negaciones llevan información de postura.
- **Lematización:** ¿mejora lo suficiente como para justificar su costo (ver tiempo medido en 2.2)? Con un corpus chico, agrupar formas flexionadas reduce la dispersión del vocabulario, pero también puede perder matices (tiempos verbales, persona).
- **Mayúsculas y acentos:** la diferencia entre `original`, `minusculas` y `sin_acentos` debería ser chica, en línea con la reducción moderada de vocabulario observada en 1.7.
- **Unigramas vs. bigramas:** ¿los bigramas ayudan en todas las variantes o solo en las que conservan las stopwords/negaciones?
- Recordar que con ≈60 comentarios *neutral* en validación, diferencias de 1–2 puntos de Macro-F1 están dentro del ruido.

### 2.6. Entrada del modelo: comentario vs. propuesta + comentario

Con el mejor preprocesamiento y n-gramas del paso anterior, se comparan las tres formas de armar la entrada descriptas en 2.2.

In [ ]:
CFG_FEAT = MEJOR_PREP["config"]["cfg_feat"]
for entrada in ["comentario", "concatenado", "separado"]:
    experimento("entrada", entrada, VARIANTE, entrada, CFG_FEAT, LR_BAL)

MEJOR_ENTRADA = mejor("entrada")
ENTRADA = MEJOR_ENTRADA["config"]["entrada"]
print("Mejor entrada:", ENTRADA)
tabla("entrada")

**Observaciones** *(completar con los resultados)*

- Como el split está agrupado por propuesta, las propuestas de validación **nunca se vieron en train**: el vocabulario del título solo puede ayudar si generaliza (por ejemplo, temas que tienden a recibir más rechazo). Es esperable que agregar la propuesta aporte poco o incluso empeore en un modelo de bolsa de palabras, que no puede modelar la *relación* entre el comentario y la propuesta (que es lo que define la postura). Esta es una limitación estructural de los enfoques clásicos, que los encoders (parte 3) y los LLMs (parte 4) sí pueden abordar al procesar ambos textos juntos.
- Comparar `concatenado` y `separado`: en `separado` las palabras del título no se mezclan con las del comentario.

### 2.7. Representación textual (features)

Se comparan distintas representaciones manteniendo el preprocesamiento y la entrada elegidos:
- **Bag-of-Words** (conteos) y **BoW binario** (presencia/ausencia).
- **TF-IDF** con unigramas, uni+bigramas y uni+bi+trigramas.
- **TF-IDF con `sublinear_tf` y `min_df=2`**: atenúa palabras muy repetidas en un mismo comentario y descarta términos que aparecen una sola vez (ruido, errores de tipeo).
- **N-gramas de caracteres** (`char_wb`, 2 a 5): robustos a errores ortográficos y variaciones morfológicas, una alternativa barata a la lematización.
- **Palabras + caracteres** combinados.

In [ ]:
CONFIGS_FEATURES = {
    "BoW 1-2":                         {"tipo": "bow", "ngram_range": (1, 2)},
    "BoW binario 1-2":                 {"tipo": "bow", "ngram_range": (1, 2), "binary": True},
    "TF-IDF 1":                        {"tipo": "tfidf", "ngram_range": (1, 1)},
    "TF-IDF 1-2":                      {"tipo": "tfidf", "ngram_range": (1, 2)},
    "TF-IDF 1-3":                      {"tipo": "tfidf", "ngram_range": (1, 3)},
    "TF-IDF 1-2 sublinear min_df=2":   {"tipo": "tfidf", "ngram_range": (1, 2), "sublinear_tf": True, "min_df": 2},
    "TF-IDF 1-3 sublinear min_df=2":   {"tipo": "tfidf", "ngram_range": (1, 3), "sublinear_tf": True, "min_df": 2},
    "Char 2-5 (char_wb)":              {"tipo": "char", "ngram_range": (2, 5), "sublinear_tf": True, "min_df": 2},
    "Char 3-6 (char_wb)":              {"tipo": "char", "ngram_range": (3, 6), "sublinear_tf": True, "min_df": 2},
    "Palabras 1-2 + char 2-5":         {"tipo": "word+char"},
}
for nombre, cfg in CONFIGS_FEATURES.items():
    experimento("features", nombre, VARIANTE, ENTRADA, cfg, LR_BAL)

MEJOR_FEAT = mejor("features")
CFG_FEAT = MEJOR_FEAT["config"]["cfg_feat"]
print("Mejores features:", MEJOR_FEAT["experimento"])
tabla("features")

**Observaciones** *(completar con los resultados)*

- ¿TF-IDF supera a BoW? Con regresión logística la normalización de TF-IDF suele ayudar porque los comentarios tienen largos muy distintos.
- ¿Los trigramas agregan algo o solo aumentan la dimensionalidad (ver `n_features`)?
- ¿Los n-gramas de caracteres compiten con las palabras? Si funcionan bien, son una forma barata de capturar morfología sin lematizar.
- Comparar el costo (`n_features`, `t_train_s`) de las representaciones más grandes contra su ganancia en Macro-F1.

### 2.8. Clasificadores e hiperparámetros

Con la representación elegida se comparan cuatro clasificadores vistos en TD VI, buscando sus hiperparámetros principales sobre validación:

| Modelo | Hiperparámetros explorados | Desbalance |
|---|---|---|
| Regresión Logística | `C` (inverso de la regularización) | `class_weight="balanced"` |
| SVM lineal (`LinearSVC`) | `C` | `class_weight="balanced"` |
| Naive Bayes complementario | `alpha` (suavizado); con TF-IDF y con conteos (BoW), que es su representación "natural" | Complement NB está pensado para clases desbalanceadas |
| Random Forest | `min_samples_leaf`, `max_features` | `class_weight="balanced_subsample"` |

In [ ]:
CFG_BOW_NB = {"tipo": "bow", "ngram_range": (1, 2)}

grilla = []
for C in [0.1, 0.3, 1, 3, 10, 30]:
    grilla.append((f"LogReg C={C}", CFG_FEAT,
                   LogisticRegression(C=C, max_iter=5000, class_weight="balanced", random_state=SEED)))
for C in [0.01, 0.03, 0.1, 0.3, 1, 3]:
    grilla.append((f"LinearSVC C={C}", CFG_FEAT,
                   LinearSVC(C=C, max_iter=10000, class_weight="balanced", random_state=SEED)))
for alpha in [0.03, 0.1, 0.3, 1.0]:
    grilla.append((f"ComplementNB alpha={alpha} | features elegidas", CFG_FEAT, ComplementNB(alpha=alpha)))
    grilla.append((f"ComplementNB alpha={alpha} | BoW 1-2", CFG_BOW_NB, ComplementNB(alpha=alpha)))
for leaf in [1, 3]:
    for mf in ["sqrt", "log2"]:
        grilla.append((f"RandomForest leaf={leaf} max_features={mf}", CFG_FEAT,
                       RandomForestClassifier(n_estimators=500, min_samples_leaf=leaf, max_features=mf,
                                              class_weight="balanced_subsample", n_jobs=-1,
                                              random_state=SEED)))

for nombre, cfg, modelo in grilla:
    experimento("modelos", nombre, VARIANTE, ENTRADA, cfg, modelo)

tabla("modelos")

In [ ]:
# Mejor configuración de cada tipo de modelo
mod = pd.DataFrame([r for r in resultados if r["grupo"] == "modelos"])
mod["tipo_modelo"] = mod["experimento"].str.split().str[0]
mejores_por_modelo = mod.loc[mod.groupby("tipo_modelo")["macro_f1"].idxmax()].sort_values("macro_f1", ascending=False)
display(mejores_por_modelo[["experimento", "macro_f1", "accuracy"] + [f"f1_{c}" for c in CLASES] + ["t_train_s"]]
        .round(3).set_index("experimento"))

# Sensibilidad a C en los modelos lineales
fig, ax = plt.subplots(figsize=(7, 4))
for tipo in ["LogReg", "LinearSVC"]:
    s = mod[mod["tipo_modelo"] == tipo].copy()
    s["C"] = s["experimento"].str.extract(r"C=([\d.]+)").astype(float)
    ax.plot(s["C"], s["macro_f1"], marker="o", label=tipo)
ax.set_xscale("log"); ax.set_xlabel("C (escala log)"); ax.set_ylabel("Macro-F1 (validación)")
ax.set_title("Sensibilidad a la regularización"); ax.legend(); plt.tight_layout(); plt.show()

#### 2.8.1. Estabilidad de la selección: validación cruzada agrupada

El conjunto de validación tiene solo ≈15% de los datos (unos 60 comentarios *neutral*), así que elegir el modelo por una única partición es ruidoso. Para la elección final, la mejor configuración de cada tipo de modelo se evalúa además con **validación cruzada de 5 folds agrupada por propuesta** (`StratifiedGroupKFold`) sobre train + validación. **El test no se usa.**

**Criterio de selección:** se elige la configuración con mayor Macro-F1 promedio en esta validación cruzada.

In [ ]:
X_cache_trval = {}

def cv_agrupada(config, n_splits=5):
    clave = (config["variante"], config["entrada"])
    if clave not in X_cache_trval:
        X_cache_trval[clave] = unir(armar_X("train", *clave), armar_X("val", *clave))
    X = X_cache_trval[clave]
    y = np.concatenate([y_train, y_val])
    grupos = np.concatenate([train_df["target"], val_df["target"]])
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = []
    for idx_tr, idx_te in sgkf.split(np.zeros(len(y)), y, groups=grupos):
        pipe = armar_pipeline(config).fit(tomar(X, idx_tr), y[idx_tr])
        scores.append(f1_score(y[idx_te], pipe.predict(tomar(X, idx_te)), average="macro"))
    return np.array(scores)

candidatos = [r for r in resultados if r["experimento"] in set(mejores_por_modelo["experimento"])
              and r["grupo"] == "modelos"]
# Se agrega también la LogReg balanceada con features por defecto (punto de partida) como referencia
candidatos.append(next(r for r in resultados if r["grupo"] == "desbalance" and "balanced" in r["experimento"]))

filas_cv = []
for r in candidatos:
    s = cv_agrupada(r["config"])
    filas_cv.append({"experimento": r["experimento"], "macro_f1_val": r["macro_f1"],
                     "macro_f1_cv_media": s.mean(), "macro_f1_cv_std": s.std()})
cv_df = pd.DataFrame(filas_cv).sort_values("macro_f1_cv_media", ascending=False).set_index("experimento")
display(cv_df.round(3))

ELEGIDO = next(r for r in candidatos if r["experimento"] == cv_df.index[0])
print("Configuración elegida:", ELEGIDO["experimento"])
print({k: v for k, v in ELEGIDO["config"].items()})

**Observaciones** *(completar con los resultados)*

- ¿Qué tan dispersos son los Macro-F1 entre folds (desvío estándar)? Si las diferencias entre modelos son menores que el desvío, no hay un ganador claro y conviene preferir el modelo más simple y barato.
- ¿El ranking por validación simple coincide con el de la validación cruzada?
- Modelos lineales (LogReg, SVM) suelen funcionar mejor que Random Forest en texto: las representaciones son de muy alta dimensión y dispersas, y cada árbol solo mira un subconjunto chico de features. Naive Bayes es el más rápido pero asume independencia entre términos.
- Limitación: los hiperparámetros de NB y RF se exploraron sobre features elegidas con regresión logística, que podrían no ser óptimas para ellos.

### 2.9. Evaluación final en test

Recién ahora se usa el conjunto de test, una única vez, para:
1. **Baseline** (TF-IDF unigramas + LogReg sin pesos), entrenado en train.
2. **Modelo elegido**, entrenado en train (mismos datos que el baseline, comparación directa).
3. **Modelo elegido reentrenado en train + validación** (configuración final: una vez fijados los hiperparámetros, la validación se puede usar como datos de entrenamiento adicionales).

Se registran los tiempos de entrenamiento e inferencia y el tamaño del modelo serializado.

In [ ]:
def evaluar_en_test(nombre, config, con_val=False):
    X_tr = armar_X("train", config["variante"], config["entrada"])
    y_tr = y_train
    if con_val:
        X_tr = unir(X_tr, armar_X("val", config["variante"], config["entrada"]))
        y_tr = np.concatenate([y_train, y_val])
    X_te = armar_X("test", config["variante"], config["entrada"])
    pipe = armar_pipeline(config)
    t0 = time.perf_counter(); pipe.fit(X_tr, y_tr); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); y_pred = pipe.predict(X_te); t_inf = time.perf_counter() - t0
    fila = {"modelo": nombre, **metricas(y_test, y_pred), "n_train": len(y_tr),
            "t_train_s": t_train, "t_inf_test_s": t_inf,
            "t_inf_por_1000_ms": t_inf / len(y_test) * 1000 * 1000,
            "tamaño_modelo_MB": len(pickle.dumps(pipe)) / 1e6,
            "usa_lematizacion": VARIANTES[config["variante"]]["lemas"]}
    return fila, pipe, y_pred

res_test = {}
for nombre, config, con_val in [
    ("Baseline (TF-IDF 1-gramas + LogReg)", BASELINE["config"], False),
    (f"Elegido: {ELEGIDO['experimento']} (train)", ELEGIDO["config"], False),
    (f"Elegido: {ELEGIDO['experimento']} (train+val)", ELEGIDO["config"], True),
]:
    res_test[nombre] = evaluar_en_test(nombre, config, con_val)

test_tabla = pd.DataFrame([f for f, _, _ in res_test.values()]).set_index("modelo")
display(test_tabla.round(3))
print(f"Nota: si el modelo usa lematización, sumar su costo de preprocesamiento "
      f"(≈{T_LEMATIZACION / len(todos) * 1000:.1f} ms por texto, medido en 2.2).")

In [ ]:
NOMBRE_BASE, NOMBRE_FINAL = list(res_test)[0], list(res_test)[-1]
_, pipe_final, y_pred_final = res_test[NOMBRE_FINAL]
_, pipe_base, y_pred_base = res_test[NOMBRE_BASE]

print("=== Modelo final (train+val) ===")
print(classification_report(y_test, y_pred_final, labels=CLASES, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (titulo, y_pred) in zip(axes, [("Baseline", y_pred_base), ("Modelo final", y_pred_final)]):
    cm = confusion_matrix(y_test, y_pred, labels=CLASES, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{titulo} — Macro-F1 = {f1_score(y_test, y_pred, average='macro'):.3f}")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real (majority_label)")
plt.suptitle("Matrices de confusión en test (normalizadas por fila)"); plt.tight_layout(); plt.show()

# F1 por clase: baseline vs. final
f1_clase = test_tabla.loc[[NOMBRE_BASE, NOMBRE_FINAL], [f"f1_{c}" for c in CLASES] + ["macro_f1"]]
f1_clase.index = ["Baseline", "Modelo final"]
ax = f1_clase.T.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("F1 (test)"); ax.set_title("F1 por clase en test"); ax.set_ylim(0, 1)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

**Observaciones** *(completar con los resultados)*

- Comparar el Macro-F1 de test con el de validación: si es similar, la selección no sobreajustó la validación.
- ¿Cuánto mejora el modelo final respecto del baseline y de dónde viene la mejora (qué clase)?
- En la matriz de confusión, ¿con qué clase se confunde más *neutral*? En la parte 1 la frontera más difusa entre anotadores era *neutral* vs. *favor*.
- ¿Reentrenar con train+val cambia mucho el resultado?

### 2.10. Análisis del modelo final

#### 2.10.1. Desempeño según el acuerdo entre anotadores

Si el modelo falla sobre todo en los comentarios donde los anotadores tampoco coincidieron, parte del error se explica por la ambigüedad de las etiquetas (techo de desempeño discutido en la parte 1).

In [ ]:
filas = []
for nivel in ["Total (3/3)", "Parcial (2/3)"]:
    m = (test_df["nivel_acuerdo"] == nivel).to_numpy()
    filas.append({"nivel_acuerdo": nivel, "n": m.sum(),
                  "accuracy": accuracy_score(y_test[m], y_pred_final[m]),
                  "macro_f1": f1_score(y_test[m], y_pred_final[m], labels=CLASES, average="macro", zero_division=0)})
display(pd.DataFrame(filas).set_index("nivel_acuerdo").round(3))

#### 2.10.2. Términos más influyentes por clase

Para modelos lineales, los coeficientes indican qué features empujan la predicción hacia cada clase. Si el modelo final no es lineal, se usa la regresión logística con las mismas features para la interpretación.

In [ ]:
pipe_interp = pipe_final
if not hasattr(pipe_final.named_steps["clf"], "coef_"):
    config_lr = dict(ELEGIDO["config"], modelo=LR_BAL)
    v, e = config_lr["variante"], config_lr["entrada"]
    pipe_interp = armar_pipeline(config_lr).fit(unir(armar_X("train", v, e), armar_X("val", v, e)),
                                                np.concatenate([y_train, y_val]))

nombres = pipe_interp.named_steps["feats"].get_feature_names_out()
clf = pipe_interp.named_steps["clf"]
top = {}
for i, clase in enumerate(clf.classes_):
    orden = np.argsort(clf.coef_[i])[::-1][:15]
    top[clase] = [f"{nombres[j]} ({clf.coef_[i][j]:.2f})" for j in orden]
display(pd.DataFrame(top)[CLASES])

#### 2.10.3. Ejemplos de errores

In [ ]:
errores = test_df.assign(pred=y_pred_final)[lambda d: d["pred"] != d["label"]]
print(f"Errores en test: {len(errores)} de {len(test_df)}\n")
for (real, pred), g in errores.groupby(["label", "pred"]):
    if len(g) < 5:
        continue
    print(f"=== Real: {real} | Predicho: {pred} ({len(g)} casos) ===")
    for _, fila in g.sample(min(2, len(g)), random_state=SEED).iterrows():
        print(f"• [{fila['target'][:60]}] {fila['comment'][:250]}")
        print(f"  Anotaciones: {fila[cols_anotadores].tolist()}\n")

**Observaciones** *(completar con los resultados)*

- **Acuerdo:** comparar el desempeño en comentarios con acuerdo total vs. parcial.
- **Términos:** ¿los términos más influyentes expresan postura (*acuerdo*, *no*, *apoyo*, *absurdo*...) o son temáticos (*madrid*, *bicis*, *barrio*...)? Si aparecen muchos términos temáticos, el modelo está aprendiendo atajos ligados a las propuestas de train, que no generalizan a propuestas nuevas.
- **Errores:** ¿los errores son comentarios irónicos, que dependen de la propuesta, o con etiquetas dudosas (anotaciones divididas)? Estos son los casos donde un modelo que entienda contexto (partes 3 y 4) podría ayudar.

### 2.11. Resumen de experimentos y guardado de resultados

Se grafica el Macro-F1 en validación de todos los experimentos y se guardan los resultados para la comparación entre enfoques (parte 5):
- `resultados_parte2_val.csv`: todos los experimentos sobre validación.
- `resultados_parte2_test.csv`: métricas, tiempos y tamaño en test del baseline y del modelo final.
- `predicciones_parte2_test.csv`: predicciones en test (para matrices de confusión y comparaciones por comentario entre enfoques).

In [ ]:
res_val = pd.DataFrame(resultados)
res_val["config"] = res_val["config"].apply(
    lambda c: f"variante={c['variante']} | entrada={c['entrada']} | feats={c['cfg_feat']} | modelo={c['modelo']}")

grupos_orden = ["referencia", "desbalance", "preprocesamiento", "entrada", "features", "modelos"]
graf = res_val[res_val["grupo"] != "referencia"].copy()
graf = graf.sort_values(["grupo", "macro_f1"], key=lambda s: s.map(grupos_orden.index) if s.name == "grupo" else s)

fig, ax = plt.subplots(figsize=(9, max(6, len(graf) * 0.22)))
sns.barplot(data=graf, y="experimento", x="macro_f1", hue="grupo", dodge=False, ax=ax)
ax.axvline(BASELINE["macro_f1"], ls="--", c="gray", label="Baseline")
ax.set_xlabel("Macro-F1 (validación)"); ax.set_ylabel(""); ax.tick_params(axis="y", labelsize=7)
ax.set_title("Todos los experimentos de la parte 2"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

res_val.to_csv("resultados_parte2_val.csv", index=False)
test_tabla.assign(t_lematizacion_por_texto_ms=T_LEMATIZACION / len(todos) * 1000).to_csv("resultados_parte2_test.csv")
pd.DataFrame({"id": test_df["id"], "label": y_test,
              "pred_baseline": y_pred_base, "pred_final": y_pred_final}).to_csv("predicciones_parte2_test.csv", index=False)
print("Archivos guardados.")

### 2.12. Conclusiones de la parte 2 *(completar con los resultados)*

- **Mejora sobre el baseline:** el Macro-F1 en test pasó de **X** (baseline) a **Y** (modelo final). La mayor parte de la mejora vino de **…** (p. ej. el balanceo de clases), mientras que **…** aportó poco.
- **Preprocesamiento:** **…** (¿conviene conservar las negaciones? ¿la lematización justifica su costo?).
- **Representación y modelo:** la mejor combinación fue **…**. Los modelos lineales **…** frente a Random Forest / Naive Bayes.
- **Costos:** entrenar el modelo final toma **≈… s** en CPU y predecir 1.000 comentarios **≈… ms**, sin GPU; el modelo ocupa **… MB**. Este es el punto de referencia de costo contra el cual comparar encoders y LLMs.
- **Limitaciones:** los modelos de bolsa de palabras no modelan la relación entre el comentario y la propuesta, ni la ironía o el contexto; y *neutral* sigue siendo la clase más difícil (F1 = **…**), en parte por la ambigüedad de las etiquetas (solo 25% de acuerdo total).